In [21]:
import requests
import pandas as pd
import re
import os


# * Listando os parâmetros que serão requisitados da API
parametros = {
    '@trimestre': "'20201'",
    '$top': 25,
    '$format': 'json',
    '$select': 'datatrimestre,valorPix,valorTED,valorTEC,valorCheque,valorBoleto,valorDOC,valorCartaoCredito,valorCartaoDebito,valorCartaoPrePago,valorTransIntrabancaria,valorConvenios,valorDebitoDireto,valorSaques,quantidadePix,quantidadeTED,quantidadeTEC,quantidadeCheque,quantidadeBoleto,quantidadeDOC,quantidadeCartaoCredito,quantidadeCartaoDebito,quantidadeCartaoPrePago,quantidadeTransIntrabancaria,quantidadeConvenios,quantidadeDebitoDireto,quantidadeSaques'
}

site = 'https://olinda.bcb.gov.br/olinda/servico/MPV_DadosAbertos/versao/v1/odata/MeiosdePagamentosTrimestralDA(trimestre=@trimestre)'


# * Requisição + Tratamento de Erro 
try:
    response = requests.get(url=site, params=parametros)
    response.raise_for_status()
    data = response.json()
    
    # * Salvando os dados em um DF
    dados_brutos = data['value']
    df = pd.DataFrame(dados_brutos)
    
    # * Tratamento de Dados
    df_copia = df.copy()
    
    
    # Função que insere sublinhado antes de maiúsculas e converte para minúsculas
    def camel_to_snake(name):
        
        # Adiciona '_' antes de maiúsculas e remove espaços extras
        s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
        return re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()
    
    # Aplicando a conversão em todas as colunas
    df_copia.columns = [camel_to_snake(col) for col in df.columns]
    
    
    # / Alterando o tipo do data_trimestre -> datetime
    df_copia = df_copia.rename(columns={'datatrimestre': 'data_trimestre'})
    df_copia['data_trimestre'] = pd.to_datetime(df_copia['data_trimestre'])
    
    # / Criando coluna Trimestre e reordenando-a
    df_copia['trimestre'] = df_copia['data_trimestre'].dt.quarter
    df_copia['data_trimestre'] =  df_copia['data_trimestre'].dt.normalize()
    
    coluna_trimestre = df_copia.pop('trimestre')
    df_copia.insert(1, 'trimestre', coluna_trimestre)
    df_copia = df_copia.sort_values(by='data_trimestre', ascending=True).reset_index(drop=True)
    
    
    # / Alterando o tipo 'quantidade' para int
    for coluna in df_copia.columns:
        if coluna[:10] == 'quantidade':
            df_copia[coluna] = pd.to_numeric(df_copia[coluna], errors='coerce').round().astype('Int64')
            print(coluna)



    display(df_copia.info())
    display(df_copia)

    # * Salvando os dados em um arquivo csv
    caminho_csv = os.path.join('..', 'data', 'stg_meios_pagamento.csv')
    df_copia.to_csv(caminho_csv, index=False, sep=';', encoding='utf-8-sig')
    
    print(f'Arquivo salvo com sucesso em: {os.path.abspath(caminho_csv)}')
    
except requests.exceptions.RequestException as erro:
    print(f'Erro ao acessar a API: {erro}')


quantidade_pix
quantidade_ted
quantidade_tec
quantidade_cheque
quantidade_boleto
quantidade_doc
quantidade_cartao_credito
quantidade_cartao_debito
quantidade_cartao_pre_pago
quantidade_trans_intrabancaria
quantidade_convenios
quantidade_debito_direto
quantidade_saques
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 28 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   data_trimestre                  25 non-null     datetime64[ns]
 1   trimestre                       25 non-null     int32         
 2   valor_pix                       25 non-null     float64       
 3   valor_ted                       25 non-null     float64       
 4   valor_tec                       25 non-null     float64       
 5   valor_cheque                    25 non-null     float64       
 6   valor_boleto                    25 non-null     float64       
 7   valor_doc  

None

,data_trimestre,trimestre,valor_pix,valor_ted,valor_tec,valor_cheque,valor_boleto,valor_doc,valor_cartao_credito,valor_cartao_debito,...,quantidade_cheque,quantidade_boleto,quantidade_doc,quantidade_cartao_credito,quantidade_cartao_debito,quantidade_cartao_pre_pago,quantidade_trans_intrabancaria,quantidade_convenios,quantidade_debito_direto,quantidade_saques
0,2020-03-31,1,0.00,6652135.13,4829.56,318844.92,1526486.76,30628.55,275659.09,171399.49,...,115238,1184511,43504,2492783,2757914,122524,296781,860764,1522197,1033408
1,2020-06-30,2,0.00,6267004.16,4647.32,237873.12,1279796.90,50881.15,224403.29,152251.89,...,85355,1175442,73764,1908028,2259651,129891,327058,752351,1503582,933086
2,2020-09-30,3,0.00,7503982.24,4159.60,259334.88,1959819.72,52156.84,279176.95,228771.89,...,86966,1389216,74378,2281876,2957016,232820,347672,776086,1623252,1007773
3,2020-12-31,4,149894.91,8036705.32,5137.42,278295.37,1919100.60,38705.14,332973.37,258456.59,...,91595,1472869,58634,2672575,3483408,350707,363890,750243,1700532,1082571
4,2021-03-31,1,625046.52,7894414.16,4352.44,253527.92,1729634.02,21777.70,309999.59,204752.91,...,79410,1389343,31687,2582363,3005005,378355,282451,766142,1704254,927769
5,2021-06-30,2,1105735.21,8566787.27,3894.81,264944.31,1839922.14,23158.84,344339.59,214381.06,...,75418,1435095,28659,2839315,3141691,543360,271290,733937,1709010,927987
6,2021-09-30,3,1556916.22,9257046.28,3684.13,271120.34,2001755.12,22306.75,398440.24,233970.03,...,80111,1497440,27668,3308864,3526063,733877,253778,774411,1765465,924190
7,2021-12-31,4,1916418.68,9807293.11,5436.19,264298.16,2070207.28,14166.99,467037.92,257458.30,...,77753,1501423,16717,3737884,3849208,928576,237060,755053,1809992,936136
8,2022-03-31,1,2067826.55,9507158.34,3865.81,255455.43,2031481.29,11976.36,452804.72,230186.87,...,71479,1494396,13195,3677384,3589968,983577,283284,786599,1910758,852739
9,2022-06-30,2,2543384.40,10449574.67,4786.14,271008.09,2209484.65,17081.53,500836.77,245563.75,...,70141,1520556,17905,3971776,3778540,1147490,273518,773501,1870657,884713


Arquivo salvo com sucesso em: c:\Users\mathe\OneDrive\Documentos\Meus Projetos\Análise de Dados\Projeto end-to-end\data\stg_meios_pagamento.csv
